# 11 — MCP Tools

        **Mode:** `services`  
        **Version:** Praval `0.8.0`  
        **Course link:** New in Praval 0.8

        This notebook makes the framework's execution visible. It records actual
        stages and renders the messages or runtime events that connect them.

        **You will see**

        - Praval launches an official-SDK stdio server without a shell.
- Streamable HTTP discovery and structured results use the same ToolSpec.
- Timeouts and session cleanup are observable.

        Run the cells from top to bottom. Committed notebooks contain no saved
        output, so everything shown is produced by your installed Praval package.


In [ ]:
from __future__ import annotations

import html as _html
import json
import os
import time
from pathlib import Path

from IPython.display import HTML, display

os.environ.setdefault("PRAVAL_OBSERVABILITY", "off")


from praval import get_provider_registry
from praval.models import ModelResponse, ProviderCapabilities


class NotebookLifecycleProvider:
    """Credential-free adapter for agents whose handlers do not call a model."""

    provider_name = "notebook-lifecycle"
    capabilities = ProviderCapabilities(text=True)

    def __init__(self, config):
        self.config = config

    def invoke(self, request, tools=None):
        return ModelResponse(
            content="Notebook lifecycle response",
            provider=self.provider_name,
            model=request.model,
        )

    def close(self):
        return None


_notebook_registry = get_provider_registry()
_notebook_registry.register_provider(
    "notebook-lifecycle",
    NotebookLifecycleProvider,
    default_model="notebook-lifecycle-model",
    capabilities=NotebookLifecycleProvider.capabilities,
)
os.environ.setdefault("PRAVAL_DEFAULT_PROVIDER", "notebook-lifecycle")
os.environ.setdefault("PRAVAL_DEFAULT_MODEL", "notebook-lifecycle-model")

EVENTS = []
_STARTED = time.perf_counter()


def stage(actor, action, detail="", spore=None):
    """Capture one observable execution stage."""
    EVENTS.append(
        {
            "ms": round((time.perf_counter() - _STARTED) * 1000, 1),
            "actor": actor,
            "action": action,
            "detail": detail,
            "spore_id": getattr(spore, "id", "") if spore else "",
        }
    )


def show_flow(*steps):
    cards = []
    for index, (name, detail) in enumerate(steps):
        if index:
            cards.append('<div style="font-size:24px;color:#557">→</div>')
        cards.append(
            '<div style="padding:12px 16px;border:1px solid #b8c7dc;'
            'border-radius:12px;background:#f7fbff;min-width:130px">'
            f'<b>{_html.escape(name)}</b><br>'
            f'<span style="color:#456;font-size:12px">'
            f'{_html.escape(detail)}</span></div>'
        )
    display(
        HTML(
            '<div style="display:flex;gap:10px;align-items:center;'
            'flex-wrap:wrap;margin:12px 0">' + "".join(cards) + "</div>"
        )
    )


def show_timeline(events=None):
    events = EVENTS if events is None else events
    rows = []
    for item in events:
        rows.append(
            "<tr>"
            f"<td>{item['ms']:.1f}</td>"
            f"<td><b>{_html.escape(str(item['actor']))}</b></td>"
            f"<td>{_html.escape(str(item['action']))}</td>"
            f"<td>{_html.escape(str(item['detail']))}</td>"
            f"<td><code>{_html.escape(str(item['spore_id'])[:12])}</code></td>"
            "</tr>"
        )
    display(
        HTML(
            '<table style="border-collapse:collapse;width:100%">'
            '<thead><tr><th>ms</th><th>actor</th><th>stage</th>'
            '<th>detail</th><th>spore</th></tr></thead><tbody>'
            + "".join(rows)
            + "</tbody></table>"
        )
    )


def show_spore(spore, label="Spore on the Reef"):
    payload = json.loads(spore.to_json())
    rendered = _html.escape(json.dumps(payload, indent=2, sort_keys=True))
    display(
        HTML(
            '<div style="border-left:5px solid #7b61ff;padding:10px 14px;'
            'background:#faf9ff;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def show_json(value, label="Result"):
    rendered = _html.escape(json.dumps(value, indent=2, sort_keys=True, default=str))
    display(
        HTML(
            '<div style="border-left:5px solid #18a999;padding:10px 14px;'
            'background:#f5fffd;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def require_env(*names):
    missing = [name for name in names if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing required notebook configuration: " + ", ".join(missing))
    return {name: os.environ[name] for name in names}


def find_example_asset(relative):
    relative = Path(relative)
    starts = [Path.cwd(), *Path.cwd().parents]
    for root in starts:
        for candidate in (root / relative, root / "examples" / relative):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Could not locate example asset: {relative}")


## Real local protocol servers

This notebook uses the official SDK server shipped with the Praval examples.
It exercises both transports supported in 0.8. Legacy SSE is intentionally not
used.


In [ ]:
import asyncio
import socket
import subprocess
import sys

from praval import Agent
from praval.mcp import MCPClient, MCPServerConfig

server_path = find_example_asset("certification/mcp_server.py")


def free_port():
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def wait_for_port(port, timeout=10):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        with socket.socket() as sock:
            sock.settimeout(0.1)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return
        time.sleep(0.05)
    raise RuntimeError("MCP Streamable HTTP server did not start")


In [ ]:
stdio_config = MCPServerConfig(
    name="notebook_stdio",
    transport="stdio",
    command=sys.executable,
    args=[str(server_path)],
    require_approval=True,
)
agent = Agent("course-mcp-agent")
async with MCPClient(stdio_config) as client:
    specs = await client.register_tools(agent)
    stage("MCP stdio", "tools discovered", f"{len(specs)} tools")
    assert all(spec.requires_approval for spec in specs)
    result = await client.call_tool(
        "notebook_stdio__echo", {"message": "praval"}
    )
    stage("MCP stdio", "tool called", result.content)
    assert result.content == "echo:praval"
    assert "notebook_stdio__echo" in agent.tools
assert not client.connected
stage("MCP stdio", "closed", "spawned process exited")
show_json(
    [spec.model_dump(exclude_none=True) for spec in specs],
    "Namespaced MCP ToolSpecs",
)


In [ ]:
port = free_port()
process = subprocess.Popen(
    [sys.executable, str(server_path), "http", str(port)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
try:
    wait_for_port(port)
    config = MCPServerConfig(
        name="notebook_http",
        transport="streamable_http",
        url=f"http://127.0.0.1:{port}/mcp",
        require_approval=False,
    )
    async with MCPClient(config) as http_client:
        http_specs = await http_client.list_tools()
        structured = await http_client.call_tool(
            "notebook_http__status", {"component": "notebook"}
        )
        stage("MCP HTTP", "structured result", structured.content)
        assert structured.metadata["structured_content"] == {
            "component": "notebook",
            "status": "ready",
        }
    assert not http_client.connected
finally:
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
    stage("MCP HTTP", "closed", "local server stopped")


In [ ]:
timeout_config = MCPServerConfig(
    name="notebook_timeout",
    transport="stdio",
    command=sys.executable,
    args=[str(server_path)],
    require_approval=False,
    tool_timeout=1.0,
)
async with MCPClient(timeout_config) as timeout_client:
    timeout_result = await timeout_client.call_tool(
        "notebook_timeout__slow", {"seconds": 2.0}
    )
    assert timeout_result.is_error
    assert "timed out" in timeout_result.content
    stage("MCP", "timeout bounded", timeout_result.content)
agent.close()
show_flow(
    ("MCP server", "tools/list"),
    ("Praval MCPClient", "namespace + schema"),
    ("Agent", "async-only ToolSpec"),
    ("tools/call", "normalized ToolResult"),
)
show_timeline()
